# Regressão linear com PIB do ano anterior

In [ ]:
import csv
from pathlib import Path
from statistics import linear_regression, mean

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
with (root / "data/raw/base_integrada.csv").open(encoding="utf-8") as file:
    gdp = {(row["codigo_municipio"], int(row["ano"])): float(row["pib_mil_reais"])
           for row in csv.DictReader(file)}

def fit(last_year):
    # Pareia apenas anos consecutivos do mesmo município.
    pairs = [(gdp[code, year - 1], value)
             for (code, year), value in gdp.items()
             if year <= last_year and (code, year - 1) in gdp]
    previous, actual = zip(*pairs)
    return linear_regression(previous, actual)

def predict(last_year):
    slope, intercept = fit(last_year)
    return {code: slope * value + intercept
            for (code, year), value in gdp.items() if year == last_year}

predicted = predict(2022)
actual = {code: value for (code, year), value in gdp.items() if year == 2023}
assert predicted.keys() == actual.keys()
errors = {code: actual[code] - predicted[code] for code in actual}
center = mean(actual.values())
r2 = 1 - sum(e**2 for e in errors.values()) / sum((y-center)**2 for y in actual.values())
mape = 100 * mean(abs(errors[code]) / actual[code] for code in actual)
print(f"2023 | {len(actual)} municípios | R² = {r2:.6f} | MAPE = {mape:.2f}%")

output = root / "outputs/regressao_linear_2024.csv"
output.parent.mkdir(exist_ok=True)
with output.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["codigo_municipio", "ano", "pib_previsto_mil_reais"])
    writer.writerows((code, 2024, round(value, 3)) for code, value in sorted(predict(2023).items()))
